## A Stylized Menu Cost Model

Consider a firm that is choosing $y$ to hit tome target variable $z$ that is moving as a random walk. You have to pay a fixed cost $c$ whenever you adjust. We can write the firm's Bellman equation as

$$ v(y,z) = (y+z)^2 + \beta \min \{ E [ \min_{y^*}v(y^*, z+\epsilon)] + c, \ E[ v(y, z+\epsilon) ] \} $$

The assumption that $z$ follows a random walk allows us to write the problem with only one state variable:
$$ v(x) = x^2 + \beta \min \{ E [ \min_{x^*}v(x^* + \epsilon)] + c, \ E[ v(x+\epsilon)  ] \} $$
where we define $x = y+z$ 

Write
$$ w(x) = E [ v(x+\epsilon)] $$
To get that
$$  v(x) = x^2 + \beta \min \{ \min_{x^*} w(x^*) + c, w(x) \} $$
Think of $w$ as pre-shock value, before you know the realization of $\epsilon$

### Backward iteration strategy
Iterate backward on $w$. Starting with a guess for $w$, call this $w_+$
1. Find the minimum to get the optimal choice of $x^*$ today for those who adjust
2. equate with $w(x^*) + c$ with $w(x)$ to find adjustment thresholds
3. Then we want to calculate today's value:

$$ A^{no-adj}(x) = E [1^{no-adj} (x+\epsilon) w_+(x+\epsilon) $$

the probability of no adjustment times the expected value conditional on no adjustment 
4. Then calculate
$$ A^{adj}(x) = E [ 1^{adj} (x+\epsilon) (W_+(x^*) + c)] $$

5. Taking expectations in the equation for $v(x)$

$$ w(x) = E [ v(x+\epsilon)] = (x-\mu)^2 + \sigma^2 + \beta [A^{no-adj}(x) + A^{adj}(x)] $$

So we will represent $w$ with degree 15 chebyshev polynomial and to integrate we will use 15 G-L nodes.

For simplicity, use QuantEcon's built-in function to get the nodes.


In [8]:
# functions to get the nodes
using QuantEcon, Distributions, BasisMatrices
μ = 0.1
σ = 1.
dist = Normal(μ, σ)
x, w = qnwlege(10, -3, 2);

function cheb_nodes(xlow, xhigh, N)
    basis = Basis(ChebParams(N, xlow, xhigh))
    return nodes(basis)[1], basis 
end

function cheb_interp(x, y, basis)
    Φ = BasisMatrix(basis, Direct(), x, 0) # Direct() does better than Expanded()
    return Φ.vals[1] \ y
end   

┌ Info: Precompiling BasisMatrices [08854c51-b66b-5062-a90d-8e7ae4547a49]
└ @ Base loading.jl:1273


cheb_interp (generic function with 1 method)

Now we will need to use Newton's methods to implement step (1) and (2), so compute the first and second derivatives of the Chebyshev representation qW

In [9]:
# Compute the $d^th$ order derivative of the Chebyshev representation:
function cheb_der(basis, x, d)
    Φ = BasisMatrix(basis, Direct(), x, d)
    return Φ.vals[1]
end

cheb_der (generic function with 1 method)

In [ ]:
function optimal_reset_and_thresholds(qW, a, b, c)
    